In [1]:
import os
import joblib
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ============================================================
# PATHS
# ============================================================

DATASET_PATH = r"C:\Users\mehar\Desktop\AI_Aquatic_Habitat_Assessment (2)\AI_Aquatic_Habitat_Assessment\ml\SEAN_FQ_Q_2011-2024_KLGO_TA.csv"

MODEL_DIR = r"C:\Users\mehar\Desktop\AI_Aquatic_Habitat_Assessment (2)\AI_Aquatic_Habitat_Assessment\ml\models"

os.makedirs(MODEL_DIR, exist_ok=True)


# ============================================================
# FEATURES AND TARGETS
# ============================================================

FEATURES = [
    "water_temperature",
    "sp_conductance",
    "turbidity"
]

TARGETS = [
    "ph",
    "do_concentration"
]


# ============================================================
# LOAD DATA
# ============================================================

print("Loading dataset...")

df = pd.read_csv(
    DATASET_PATH,
    low_memory=False
)

print("Original rows:", len(df))


# ============================================================
# CLEAN DATA
# ============================================================

df["timestamp_utc"] = pd.to_datetime(
    df["timestamp_utc"],
    errors="coerce",
    utc=True
)

for col in FEATURES + TARGETS:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

df = df.dropna(
    subset=["timestamp_utc"] + FEATURES + TARGETS
)

print("Rows after removing missing values:", len(df))


# ============================================================
# PHYSICAL VALIDITY FILTER
# ============================================================

df = df[
    (df["water_temperature"] >= 0) &
    (df["water_temperature"] <= 40) &
    (df["sp_conductance"] >= 0) &
    (df["turbidity"] >= 0) &
    (df["ph"] >= 0) &
    (df["ph"] <= 14) &
    (df["do_concentration"] >= 0)
].copy()

print("Rows after physical filtering:", len(df))


# ============================================================
# SORT CHRONOLOGICALLY
# ============================================================

df = df.sort_values("timestamp_utc").reset_index(drop=True)


# ============================================================
# CHRONOLOGICAL 80/20 SPLIT
# ============================================================

split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

print("\n================ TIME SPLIT ================")

print(
    "Training period:",
    train_df["timestamp_utc"].min(),
    "to",
    train_df["timestamp_utc"].max()
)

print(
    "Testing period:",
    test_df["timestamp_utc"].min(),
    "to",
    test_df["timestamp_utc"].max()
)

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))


X_train = train_df[FEATURES]
X_test = test_df[FEATURES]


# ============================================================
# GRADIENT BOOSTING PARAMETERS
# ============================================================

MODEL_PARAMETERS = {
    "n_estimators": 200,
    "learning_rate": 0.1,
    "max_depth": 3,
    "random_state": 42
}


# ============================================================
# TRAIN pH MODEL
# ============================================================

print("\n============================================")
print("TRAINING FINAL pH MODEL")
print("============================================")

y_train_ph = train_df["ph"]
y_test_ph = test_df["ph"]

ph_model = GradientBoostingRegressor(
    **MODEL_PARAMETERS
)

ph_model.fit(
    X_train,
    y_train_ph
)

ph_predictions = ph_model.predict(X_test)

ph_mae = mean_absolute_error(
    y_test_ph,
    ph_predictions
)

ph_rmse = mean_squared_error(
    y_test_ph,
    ph_predictions
) ** 0.5

ph_r2 = r2_score(
    y_test_ph,
    ph_predictions
)

print("pH MAE :", round(ph_mae, 4))
print("pH RMSE:", round(ph_rmse, 4))
print("pH R²  :", round(ph_r2, 4))


# ============================================================
# TRAIN DO MODEL
# ============================================================

print("\n============================================")
print("TRAINING FINAL DO MODEL")
print("============================================")

y_train_do = train_df["do_concentration"]
y_test_do = test_df["do_concentration"]

do_model = GradientBoostingRegressor(
    **MODEL_PARAMETERS
)

do_model.fit(
    X_train,
    y_train_do
)

do_predictions = do_model.predict(X_test)

do_mae = mean_absolute_error(
    y_test_do,
    do_predictions
)

do_rmse = mean_squared_error(
    y_test_do,
    do_predictions
) ** 0.5

do_r2 = r2_score(
    y_test_do,
    do_predictions
)

print("DO MAE :", round(do_mae, 4), "mg/L")
print("DO RMSE:", round(do_rmse, 4), "mg/L")
print("DO R²  :", round(do_r2, 4))


# ============================================================
# FEATURE IMPORTANCE
# ============================================================

print("\n============================================")
print("FEATURE IMPORTANCE")
print("============================================")

print("\npH model:")
for feature, importance in zip(
    FEATURES,
    ph_model.feature_importances_
):
    print(
        f"{feature}: {importance:.4f}"
    )

print("\nDO model:")
for feature, importance in zip(
    FEATURES,
    do_model.feature_importances_
):
    print(
        f"{feature}: {importance:.4f}"
    )


# ============================================================
# SAVE MODELS
# ============================================================

ph_model_path = os.path.join(
    MODEL_DIR,
    "final_ph_gradient_boosting.pkl"
)

do_model_path = os.path.join(
    MODEL_DIR,
    "final_do_gradient_boosting.pkl"
)

joblib.dump(
    ph_model,
    ph_model_path
)

joblib.dump(
    do_model,
    do_model_path
)


# ============================================================
# SAVE METRICS
# ============================================================

metrics = pd.DataFrame([
    {
        "target": "pH",
        "model": "Gradient Boosting",
        "MAE": ph_mae,
        "RMSE": ph_rmse,
        "R2": ph_r2
    },
    {
        "target": "DO",
        "model": "Gradient Boosting",
        "MAE": do_mae,
        "RMSE": do_rmse,
        "R2": do_r2
    }
])

metrics_path = os.path.join(
    MODEL_DIR,
    "final_model_metrics.csv"
)

metrics.to_csv(
    metrics_path,
    index=False
)


print("\n============================================")
print("FINAL MODELS SAVED")
print("============================================")

print(ph_model_path)
print(do_model_path)
print(metrics_path)

print("\nFinal model training completed successfully!")

Loading dataset...
Original rows: 61552
Rows after removing missing values: 56501
Rows after physical filtering: 56276

================ TIME SPLIT ================
Training period: 2011-04-25 23:00:51+00:00 to 2022-07-31 17:00:00+00:00
Testing period: 2022-07-31 18:00:00+00:00 to 2024-10-24 21:00:00+00:00
Training rows: 45020
Testing rows: 11256

TRAINING FINAL pH MODEL
pH MAE : 0.0683
pH RMSE: 0.0896
pH R²  : 0.6087

TRAINING FINAL DO MODEL
DO MAE : 0.3611 mg/L
DO RMSE: 0.4627 mg/L
DO R²  : 0.2344

FEATURE IMPORTANCE

pH model:
water_temperature: 0.0345
sp_conductance: 0.9138
turbidity: 0.0517

DO model:
water_temperature: 0.6674
sp_conductance: 0.1588
turbidity: 0.1738

FINAL MODELS SAVED
C:\Users\mehar\Desktop\AI_Aquatic_Habitat_Assessment (2)\AI_Aquatic_Habitat_Assessment\ml\models\final_ph_gradient_boosting.pkl
C:\Users\mehar\Desktop\AI_Aquatic_Habitat_Assessment (2)\AI_Aquatic_Habitat_Assessment\ml\models\final_do_gradient_boosting.pkl
C:\Users\mehar\Desktop\AI_Aquatic_Habitat_A